In [4]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

In [5]:
def load_runbook(path):
    """Parse a NeurIPS Big-ANN streaming runbook YAML into a DataFrame of steps."""
    with open(path) as f:
        rb = yaml.safe_load(f)

    # exactly one top-level dataset key (matches postgres_run_streaming_runbook.py)
    assert len(rb) == 1, f"expected 1 dataset key, got {list(rb)}"
    dataset = next(iter(rb))
    body = rb[dataset]

    max_pts = body.get("max_pts")
    rows = []
    for k, v in body.items():
        if not isinstance(k, int):     # skip 'max_pts' and other metadata
            continue
        op = str(v["operation"]).strip().strip("'\"")
        start = v.get("start")
        end = v.get("end")
        rows.append({
            "step": k,
            "operation": op,
            "start": start,
            "end": end,
            "batch_size": (end - start) if (start is not None and end is not None) else 0,
        })

    df = pd.DataFrame(rows).sort_values("step").reset_index(drop=True)
    df.attrs["dataset"] = dataset
    df.attrs["max_pts"] = max_pts
    return df

In [13]:
def search_scale_factor(df, target_search_fraction, queries_per_search=10_000):
    """
    Factor to scale total search volume so searches make up
    `target_search_fraction` of all vector ops, with inserts+deletes held fixed.

    target_search_fraction: search share of (insert + delete + search) vectors.
        10% searches / 90% updates -> 0.10
        50% searches / 50% updates -> 0.50

    Returns the multiplier on the current search volume
    (current = n_search_steps * queries_per_search).
    """
    vol = df.groupby("operation")["batch_size"].sum()
    updates = int(vol.get("insert", 0)) + int(vol.get("delete", 0))   # fixed
    search_base = int((df["operation"] == "search").sum()) * queries_per_search

    p = target_search_fraction
    if not 0.0 < p < 1.0:
        raise ValueError("target_search_fraction must be in (0, 1)")
    if search_base == 0:
        raise ValueError("runbook has no search steps to scale")

    target_search = p / (1.0 - p) * updates        # search : updates = p : (1-p)
    factor = target_search / search_base

    # print(f"updates (insert+delete), fixed: {updates:>14,}")
    # print(f"current search volume:          {search_base:>14,}")
    # print(f"target search volume:           {target_search:>14,.0f}")
    # print(f"-> scale searches by {factor:.4f}x "
    #       f"(achieves {p:.0%} searches / {1-p:.0%} updates)")
    return factor

In [14]:
runbooks = {
    "msturing-500M-shift":     "/dataset/big-ann-benchmarks/data/MSTuring-500M-shift/runbook_msturing500Mshift.yaml",
    "bigann-500M-clustered": "/dataset/big-ann-benchmarks/data/bigann-500M-clustered/runbook_bigann-500M-clustered.yaml"
}

In [15]:
for runbook in runbooks:
    df = load_runbook(runbooks[runbook])
    print(f"dataset: {runbook}=========================")
    factor5_95 = search_scale_factor(df, 0.05)
    factor50_50 = search_scale_factor(df, 0.50)
    factor95_5 = search_scale_factor(df, 0.95)
    print(f"5% searches / 95% updates factor: {factor5_95}")
    print(f"50% searches / 50% updates factor: {factor50_50}")
    print(f"95% searches / 5% updates factor: {factor95_5}")

dataset: msturing-500M-shift=========================
5% searches / 95% updates factor: 14.7610508202324
50% searches / 50% updates factor: 280.45996558441556
95% searches / 5% updates factor: 5328.739346103891
dataset: bigann-500M-clustered=========================
5% searches / 95% updates factor: 7.930393067434212
50% searches / 50% updates factor: 150.67746828125
95% searches / 5% updates factor: 2862.8718973437476


In [16]:
def get_timing_stats_gpann(gpann_df, name="GP-ANN", nprobe=6):
    """
    Compute timing and recall statistics for a GP-ANN run.
    
    Args:
        gpann_df: pd.DataFrame
            GP-ANN results DataFrame with 'operation', 'time_s', 'nprobe', 'recall@10' columns.
        name: str
            Name/label for the run (for output).
        nprobe: int
            NProbe parameter to filter search timing and recall (default: 6).
    
    Returns:
        dict with keys:
            - 'total_time_s': float, total time of operations
            - 'by_operation': dict mapping operation -> {'time_s': float, 'pct': float}
            - 'avg_recall': float, average recall@10 for the specified nprobe
            - 'summary_str': str, formatted summary for printing
    """
    # Filter to include all non-SEARCH operations, plus SEARCH operations with specific nprobe
    df = gpann_df.copy()
    search_mask = df['operation'] == 'SEARCH'
    if 'nprobe' in df.columns:
        search_mask = search_mask & (df['nprobe'] == nprobe)
    
    # Keep all non-search operations and filtered SEARCH operations
    df = pd.concat([df[~df['operation'].isin(['SEARCH'])], df[search_mask]])
    
    # Group by operation and sum the time
    timing_by_op = df.groupby('operation')['time_s'].sum()
    
    total_time = timing_by_op.sum()
    
    # Compute percentages
    pct_by_op = (timing_by_op / total_time * 100) if total_time > 0 else timing_by_op * 0
    
    # Compute average recall for specific nprobe (already filtered above)
    search_df = df[df['operation'] == 'SEARCH']
    
    recall_col = 'recall@10'
    avg_recall = None
    if not search_df.empty and recall_col in search_df.columns:
        # Filter out invalid recall values
        valid_recalls = search_df[(search_df[recall_col] != 'N/A') & (search_df[recall_col].notna())][recall_col]
        if not valid_recalls.empty:
            avg_recall = pd.to_numeric(valid_recalls, errors='coerce').mean()
    
    # Build result dict
    result = {
        'total_time_s': total_time,
        'by_operation': {},
        'avg_recall': avg_recall,
        'name': name
    }
    
    for op in timing_by_op.index:
        result['by_operation'][op] = {
            'time_s': timing_by_op[op],
            'pct': pct_by_op[op]
        }
    
    # Build formatted summary string
    lines = [f"\n{'='*60}"]
    lines.append(f"Timing Statistics: {name}")
    lines.append(f"{'='*60}")
    lines.append(f"{'Operation':<15} {'Time (s)':<15} {'Percentage':<15}")
    lines.append(f"{'-'*45}")
    
    for op in sorted(result['by_operation'].keys()):
        time_val = result['by_operation'][op]['time_s']
        pct_val = result['by_operation'][op]['pct']
        lines.append(f"{op:<15} {time_val:>12.2f}s  {pct_val:>12.1f}%")
    
    lines.append(f"{'-'*45}")
    lines.append(f"{'TOTAL':<15} {total_time:>12.2f}s  {100.0:>12.1f}%")
    
    if avg_recall is not None:
        lines.append(f"\n'Average Recall@10 (nprobe={nprobe})':<30 {avg_recall:>12.4f}")
    
    lines.append(f"{'='*60}\n")
    
    result['summary_str'] = '\n'.join(lines)
    
    return result

In [17]:
def get_timing_stats(surge_df, name="SURGE", routing_mode="NProbe", routing_param=5,
                     search_fraction=None):
    """
    Compute timing and recall statistics for a SURGE run.

    Timing per operation uses 'time_s', except rebuild rows, which use
    'dorebuild_wall_s' (the non-overlappable rebuild execution; 'time_s' for
    rebuilds also includes repartition planning that can run concurrently with
    queries/inserts).

    Args:
        surge_df: pd.DataFrame
            SURGE results DataFrame with 'operation' and 'time_s' columns.
        name: str
            Name/label for the run (for output).
        routing_mode: str
            Routing mode to filter search timing and recall (default: "NProbe").
        routing_param: int
            Routing parameter to filter search timing and recall (default: 5).
        search_fraction: float or None
            Which --search-fraction variant to keep. Required when the run swept
            multiple fractions, otherwise search time is summed across variants.

    Returns:
        dict with keys:
            - 'total_time_s': float, total time of operations
            - 'by_operation': dict mapping operation -> {'time_s': float, 'pct': float}
            - 'avg_recall': float, average recall@10 for the specified routing mode/param
            - 'summary_str': str, formatted summary for printing
    """
    # Filter searches down to ONE variant so each search step is counted once.
    # Search rows are emitted per (mode, param) AND per search_fraction; without
    # pinning all of them the groupby sum multi-counts the same step.
    df = surge_df.copy()
    search_mask = df['operation'] == 'search'
    if 'mode' in df.columns and 'param' in df.columns:
        search_mask = (
            search_mask
            & (df['mode'] == routing_mode)
            & np.isclose(pd.to_numeric(df['param'], errors='coerce'), routing_param)
        )
    if 'search_fraction' in df.columns:
        is_search = df['operation'] == 'search'
        fracs = sorted(pd.to_numeric(df.loc[is_search, 'search_fraction'],
                                     errors='coerce').dropna().unique())
        if search_fraction is not None:
            search_mask = search_mask & np.isclose(
                pd.to_numeric(df['search_fraction'], errors='coerce'), search_fraction)
        elif len(fracs) > 1:
            raise ValueError(
                f"Multiple search_fraction variants present {fracs}; "
                f"pass search_fraction=<one of them> to avoid multi-counting search time.")

    # Keep non-search operations and filtered search operations
    df = pd.concat([df[~df['operation'].isin(['search'])], df[search_mask]])

    # Per-operation timing: use time_s, but rebuild rows use dorebuild_wall_s
    # (non-overlappable rebuild execution; time_s also includes repartition
    # planning that can overlap with queries/inserts).
    df['_time'] = df['time_s']
    if 'dorebuild_wall_s' in df.columns:
        rebuild_mask = df['operation'] == 'rebuild'
        df.loc[rebuild_mask, '_time'] = df.loc[rebuild_mask, 'dorebuild_wall_s']

    # Group by operation and sum the time
    timing_by_op = df.groupby('operation')['_time'].sum()

    total_time = timing_by_op.sum()

    # Compute percentages
    pct_by_op = (timing_by_op / total_time * 100) if total_time > 0 else timing_by_op * 0

    # Compute average recall for specific routing mode and param (already filtered above)
    search_df = df[df['operation'] == 'search']

    recall_col = 'recall@10'
    avg_recall = None
    if not search_df.empty and recall_col in search_df.columns:
        # Filter out invalid recall values (-1)
        valid_recalls = search_df[search_df[recall_col] >= 0][recall_col]
        if not valid_recalls.empty:
            avg_recall = valid_recalls.mean()

    # Build result dict
    result = {
        'total_time_s': total_time,
        'by_operation': {},
        'avg_recall': avg_recall,
        'name': name
    }

    for op in timing_by_op.index:
        result['by_operation'][op] = {
            'time_s': timing_by_op[op],
            'pct': pct_by_op[op]
        }

    # Build formatted summary string
    lines = [f"\n{'='*60}"]
    lines.append(f"Timing Statistics: {name}")
    lines.append(f"{'='*60}")
    lines.append(f"{'Operation':<15} {'Time (s)':<15} {'Percentage':<15}")
    lines.append(f"{'-'*45}")

    for op in sorted(result['by_operation'].keys()):
        time_val = result['by_operation'][op]['time_s']
        pct_val = result['by_operation'][op]['pct']
        lines.append(f"{op:<15} {time_val:>12.2f}s  {pct_val:>12.1f}%")

    lines.append(f"{'-'*45}")
    lines.append(f"{'TOTAL':<15} {total_time:>12.2f}s  {100.0:>12.1f}%")

    if avg_recall is not None:
        label = f"Average Recall@10 (mode={routing_mode}, param={routing_param})"
        lines.append(f"\n{label:<50}{avg_recall:>12.4f}")

    lines.append(f"{'='*60}\n")

    result['summary_str'] = '\n'.join(lines)

    return result

In [20]:
def remove_duplicate_steps(df):
    # df = df.sort_values(by=["step", "operation"], ascending=[True, True])
    # df = df.drop_duplicates(subset=["step"], keep="last")
    # return df
    # ^ dropped all the steps that has multiple search entries
    # drop only is the step, operation, param, mode are the same
    df = df.sort_values(by=["step", "operation", "param", "mode"], ascending=[True, True, True, True])
    df = df.drop_duplicates(subset=["step", "operation", "param", "mode"], keep="last")
    return df

# MSturing-500M-shift

In [21]:
surge_msturing500Mshiftt10000_path = "/users/dkhimey/surge/results/msturing-500M-shift_t10000/results.csv"
surge_msturing500Mshiftt10000 = pd.read_csv(surge_msturing500Mshiftt10000_path)

surge_msturing500Mshiftt6000_path = "/users/dkhimey/surge/results/msturing-500M-shift_t6000/results.csv"
surge_msturing500Mshiftt6000 = pd.read_csv(surge_msturing500Mshiftt6000_path)

gpann_msturing500Mshift = pd.read_csv("/dataset/big-ann-benchmarks/data/MSTuring-500M-shift/gpann_partitions/msturing500Mshift_results.csv")

surge_msturing500Mshiftt6000 = remove_duplicate_steps(surge_msturing500Mshiftt6000)
surge_msturing500Mshiftt10000 = remove_duplicate_steps(surge_msturing500Mshiftt10000)

In [23]:
max_step=300

stats_surge_msturing500Mshiftt10000 = get_timing_stats(surge_msturing500Mshiftt10000[surge_msturing500Mshiftt10000["step"]<max_step], routing_mode="RecallTarget", routing_param=.9)
print(stats_surge_msturing500Mshiftt10000['summary_str'])

stats_surge_msturing500Mshiftt6000 = get_timing_stats(surge_msturing500Mshiftt6000[surge_msturing500Mshiftt6000["step"]<max_step], routing_mode="RecallTarget", routing_param=.9)
print(stats_surge_msturing500Mshiftt6000['summary_str'])

stats_gpann_msturing500Mshift = get_timing_stats_gpann(gpann_msturing500Mshift[gpann_msturing500Mshift["step"]<max_step], nprobe=9)
print(stats_gpann_msturing500Mshift["summary_str"])


Timing Statistics: SURGE
Operation       Time (s)        Percentage     
---------------------------------------------
delete                 53.86s           1.0%
insert               5340.34s          97.6%
search                 76.08s           1.4%
---------------------------------------------
TOTAL                5470.27s         100.0%

Average Recall@10 (mode=RecallTarget, param=0.9)        0.8501


Timing Statistics: SURGE
Operation       Time (s)        Percentage     
---------------------------------------------
delete                104.48s           0.9%
insert               6093.78s          50.0%
rebuild              5901.99s          48.4%
search                 82.98s           0.7%
---------------------------------------------
TOTAL               12183.23s         100.0%

Average Recall@10 (mode=RecallTarget, param=0.9)        0.8481


Timing Statistics: GP-ANN
Operation       Time (s)        Percentage     
---------------------------------------------
BUILD       